In [1]:
# ==============================================================================
# CELL 1: CÀI ĐẶT MÔI TRƯỜNG THƯ VIỆN 
# ==============================================================================
!pip install -q -U transformers==4.38.0
!pip install -q -U peft==0.9.0
!pip install -q -U accelerate==0.27.0
!pip install -q -U datasets==2.17.0
!pip install -q -U sentencepiece 

print("✅ Đã cài đặt xong các thư viện cần thiết!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.7/279.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 12.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into acc

In [2]:
# ==============================================================================
# CELL 2: NANO-REASON - KỊCH BẢN A (STANDARD LORA BASELINE)
# Học cào bằng (Cross-Entropy tiêu chuẩn), gộp 3 tập dữ liệu, fix lỗi FP16.
# ==============================================================================
import os
import gc
import json
import torch
import random
import numpy as np
from pathlib import Path
from typing import List, Dict
from dataclasses import dataclass
from enum import Enum
from torch.utils.data import ConcatDataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, TaskType, get_peft_model

# ------------------------------------------------------------------------------
# 1. TỐI ƯU PHẦN CỨNG & CỐ ĐỊNH HẠT GIỐNG (SEED)
# ------------------------------------------------------------------------------
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "true"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)
print("✅ Đã thiết lập môi trường và cố định Seed = 42!")

# ------------------------------------------------------------------------------
# 2. CẤU TRÚC DỮ LIỆU & DATASET (Trả về Text thô, KHÔNG CÓ step_labels)
# ------------------------------------------------------------------------------
class StepType(Enum):
    UNDERSTAND = 0; PLAN = 1; EXECUTE = 2; VERIFY = 3
    @property
    def tag(self) -> str: return f"#### {self.name} ####"

@dataclass
class ReasoningStep:
    step_type: StepType
    content: str

@dataclass
class ReasoningChain:
    question: str
    steps: Dict[StepType, ReasoningStep]
    ground_truth: str
    
    def get_full_text(self) -> str:
        text = f"Question: {self.question}\n\nSolution:\n"
        for stype in [StepType.UNDERSTAND, StepType.PLAN, StepType.EXECUTE, StepType.VERIFY]:
            text += f"{stype.tag}\n{self.steps[stype].content}\n\n"
        text += f"Final Answer: {self.ground_truth}"
        return text

class StepAwareDataset(torch.utils.data.Dataset):
    def __init__(self, data_path: Path, tokenizer, max_length: int = 1024):
        self.data_path = data_path
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.tokenizer.padding_side = "right"
        self.chains = self._load_data()

    def _load_data(self) -> List[ReasoningChain]:
        chains = []
        if not self.data_path.exists():
            print(f"⚠️ Không tìm thấy file: {self.data_path.name}")
            return chains
            
        with open(self.data_path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                data = json.loads(line)
                if not data.get('is_correct', True) or 'cot_4steps' not in data: continue
                cot = data['cot_4steps']
                if not all(k in cot for k in ['understand', 'plan', 'execute', 'verify']): continue
                    
                chain = ReasoningChain(
                    question=data.get('question', ''),
                    steps={
                        StepType.UNDERSTAND: ReasoningStep(StepType.UNDERSTAND, cot['understand']),
                        StepType.PLAN: ReasoningStep(StepType.PLAN, cot['plan']),
                        StepType.EXECUTE: ReasoningStep(StepType.EXECUTE, cot['execute']),
                        StepType.VERIFY: ReasoningStep(StepType.VERIFY, cot['verify'])
                    },
                    ground_truth=data.get('ground_truth', '')
                )
                chains.append(chain)
        return chains

    def __len__(self) -> int: return len(self.chains)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        full_text = self.chains[idx].get_full_text()
        encoding = self.tokenizer(
            full_text, max_length=self.max_length, truncation=True, padding='max_length', return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0)
        }

# ------------------------------------------------------------------------------
# 3. TẢI TOKENIZER & GỘP 3 TẬP DỮ LIỆU
# ------------------------------------------------------------------------------
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
DATA_DIR = Path("/kaggle/input/datasets/trinhduc041/nckh-processed-data")

print(f"\n⏳ Đang tải Tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("⏳ Đang tải và gộp 3 tập Dataset (GSM8K, MATH, VNHSGE)...")
gsm8k_train_dataset = StepAwareDataset(DATA_DIR / "gsm8k_train_processed.jsonl", tokenizer, 1024)
math_train_dataset = StepAwareDataset(DATA_DIR / "math_train_processed.jsonl", tokenizer, 1024)
vnhsge_dataset = StepAwareDataset(DATA_DIR / "vnhsge_processed.jsonl", tokenizer, 1024)

dataset_list = []
if len(gsm8k_train_dataset) > 0: dataset_list.append(gsm8k_train_dataset)
if len(math_train_dataset) > 0: dataset_list.append(math_train_dataset)
if len(vnhsge_dataset) > 0: dataset_list.append(vnhsge_dataset)

full_train_dataset = ConcatDataset(dataset_list)
print(f"✅ Đã gộp thành công! Tổng số mẫu huấn luyện: {len(full_train_dataset):,}")

# ------------------------------------------------------------------------------
# 4. CẤU HÌNH LORA & TẢI MÔ HÌNH (CÓ BẢN VÁ LỖI FP16)
# ------------------------------------------------------------------------------
class LoRAConfigManager:
    def __init__(self, lora_alpha: int = 32, lora_dropout: float = 0.05):
        self.lora_alpha = lora_alpha
        self.lora_dropout = lora_dropout
        self.target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    
    def create_config(self, rank: int = 16) -> LoraConfig:
        return LoraConfig(
            r=rank, lora_alpha=self.lora_alpha, target_modules=self.target_modules,
            lora_dropout=self.lora_dropout, bias="none", task_type=TaskType.CAUSAL_LM
        )

print("\n⏳ Đang tải Base Model (Qwen2.5-3B) lên GPU...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
)
base_model.gradient_checkpointing_enable()
base_model.enable_input_require_grads()

lora_manager = LoRAConfigManager()
default_lora_config = lora_manager.create_config(rank=16)

print("⏳ Đang bọc Adapter LoRA (Rank 16)...")
model = get_peft_model(base_model, default_lora_config)

# 🛑 BẢN VÁ LỖI TẠI ĐÂY: Ép tham số LoRA về float32 để GradScaler hoạt động
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.print_trainable_parameters()

# ------------------------------------------------------------------------------
# 5. KÍCH HOẠT HUẤN LUYỆN (BASELINE)
# ------------------------------------------------------------------------------
print("\n" + "="*80)
print("🚀 KHỞI ĐỘNG HUẤN LUYỆN: KỊCH BẢN A (STANDARD LORA)")
print("="*80)

output_dir_A = Path('/kaggle/working/Standard_LoRA')
output_dir_A.mkdir(parents=True, exist_ok=True)

# Vũ khí cốt lõi: Tự động tính CrossEntropy, phớt lờ 4 bước
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args_A = TrainingArguments(
    output_dir=str(output_dir_A),
    per_device_train_batch_size=2,          
    gradient_accumulation_steps=4,          
    learning_rate=2e-4,                     
    num_train_epochs=1,    # Chạy 1 Epoch trên toàn bộ data                   
    logging_steps=10,                       
    save_strategy="epoch",                  
    fp16=True,                              
    report_to="tensorboard",                
    optim="adamw_torch",   # Chuẩn PyTorch, không dùng bitsandbytes
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    remove_unused_columns=False,
)

trainer_A = Trainer(
    model=model,
    train_dataset=full_train_dataset, 
    args=training_args_A,
    data_collator=data_collator,
)

torch.cuda.empty_cache()
gc.collect()

print("⏳ Đang tiến hành huấn luyện Kịch bản A...")
trainer_A.train()

print("\n🎉 ĐÃ HOÀN THÀNH KỊCH BẢN A!")
print(f"📁 Checkpoint và log được lưu tại: {output_dir_A}")

2026-03-09 11:24:09.361515: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773055449.571670      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773055449.631420      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773055450.127897      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773055450.127944      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773055450.127948      24 computation_placer.cc:177] computation placer alr

✅ Đã thiết lập môi trường và cố định Seed = 42!

⏳ Đang tải Tokenizer: Qwen/Qwen2.5-3B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

⏳ Đang tiến hành huấn luyện Kịch bản A...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
10,0.674900
20,0.513000
30,0.369000
40,0.275000
50,0.225300
60,0.235100
70,0.234900
80,0.226800
90,0.223100
100,0.238400


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



🎉 ĐÃ HOÀN THÀNH KỊCH BẢN A!
📁 Checkpoint và log được lưu tại: /kaggle/working/Standard_LoRA


In [3]:
# ==============================================================================
# CELL 3: XUẤT FILE LOG SANG CSV ĐỂ VẼ BIỂU ĐỒ BÁO CÁO NCKH
# ==============================================================================
import json
import pandas as pd
from pathlib import Path

# Tìm file trainer_state.json mới nhất
output_dir = Path('/kaggle/working/Standard_LoRA')
state_files = list(output_dir.rglob('trainer_state.json'))

if state_files:
    # Lấy file của thư mục checkpoint cuối cùng
    latest_state_file = sorted(state_files, key=lambda x: x.stat().st_mtime)[-1]
    
    with open(latest_state_file, 'r') as f:
        data = json.load(f)
        
    # Lọc lấy các bước có ghi nhận 'loss'
    log_history = [log for log in data['log_history'] if 'loss' in log]
    
    if log_history:
        df = pd.DataFrame(log_history)
        # Sắp xếp lại các cột cho gọn gàng
        cols = ['step', 'epoch', 'loss', 'learning_rate']
        cols = [c for c in cols if c in df.columns] + [c for c in df.columns if c not in cols]
        df = df[cols]
        
        csv_path = output_dir / 'baseline_metrics.csv'
        df.to_csv(csv_path, index=False)
        print(f"✅ Đã xuất thành công file CSV tại: {csv_path}")
        print("   Bạn có thể dùng file này để vẽ đường cong hội tụ (Convergence Curve) đọ với Kịch bản Đề xuất!")
    else:
        print("⚠️ Không tìm thấy dữ liệu 'loss' trong file log.")
else:
    print("⚠️ Không tìm thấy file trainer_state.json nào.")

✅ Đã xuất thành công file CSV tại: /kaggle/working/Standard_LoRA/baseline_metrics.csv
   Bạn có thể dùng file này để vẽ đường cong hội tụ (Convergence Curve) đọ với Kịch bản Đề xuất!
